# 춘천시 카드 데이터 전처리 및 기초통계

## 1. 실행 환경 및 입력 경로 설정
분석에 필요한 라이브러리와 팀원 전처리 함수를 불러오고, 데이터1 파일의 경로와 존재 여부를 확인한다.

In [ ]:
from pathlib import Path
import pandas as pd
from prepare_card import prepare

path1 = Path(
    r"경로 입력"
    )

if not path1.is_file():
    raise FileNotFoundError("데이터1 파일 경로를 확인해 주세요.")

## 2. 원본 데이터 품질 점검
전체 원본의 결측값·빈 문자열·결측 의심 문자열을 점검한다. 춘천시 행과 법인 제외 내역을 확인하고, 춘천시 개인카드의 원본 집계 키 중복 및 앞뒤 공백 여부를 검사한다.

검사한 문자열 외의 비공개 표시와 행 생략 규칙은 별도 확인이 필요하다.

In [ ]:
# 점검용 원본 읽기
raw1 = pd.read_csv(
    path1, sep="\t", encoding="cp949",
    dtype=str, keep_default_na=False
)

raw1.columns = raw1.columns.str.strip()

# 공백을 제거한 점검용 복사본
checked1 = raw1.apply(lambda col: col.str.strip())

# 1. 전체 컬럼 값 점검
value_check = pd.DataFrame({
    "실제 결측": checked1.isna().sum(),
    "빈 문자열": checked1.eq("").sum(),
    "결측 의심 문자열": checked1.apply(
        lambda col: col.str.upper().isin(
            ["NULL", "NONE", "NAN", "NA", "N/A"]
        )
    ).sum()
})

display(value_check)

# 2. 춘천시 선택·법인 제외 내역
chuncheon = checked1["MCT_SGG_CD"].eq("강원 춘천시")
corporate = (
    checked1["AGE_CCD"].eq("법인")
    | checked1["SEX_CCD"].eq("법인")
)

print("전체 행 수:", len(checked1))
print("춘천시 행 수:", int(chuncheon.sum()))
print("춘천시 법인 행 수:", int((chuncheon & corporate).sum()))

personal1 = checked1.loc[chuncheon & ~corporate].copy()
print("춘천시 개인카드 행 수:", len(personal1))

# 3. 연령 통합 전 원본 집계 키 중복
raw_keys = [
    "TA_YMD", "TIME_GB", "MCT_SGG_CD",
    "MCT_RY_CD", "SEX_CCD", "AGE_CCD"
]

print(
    "춘천시 개인카드 집계 키 중복 행 수:",
    int(personal1.duplicated(raw_keys, keep=False).sum())
)

# 팀원 코드는 공백 정리 없이 필터링하므로 차이가 있는지 확인
changed = raw1.ne(checked1).fillna(False).any(axis=1)
print("공백 정리로 값이 달라진 행 수:", int(changed.sum()))

## 3. 깃헙 코드로 전처리 실행
`prepare_card.py`를 사용해 춘천시 개인카드 데이터를 처리한다. 60~90대는 ‘60대 이상’으로 통합하고, 시간대와 성별을 합산해 날짜×지역×업종×연령별 결제금액·결제건수를 저장한다.
월별 연령 요약도 함께 생성되며, 재실행 시 기존 결과 CSV를 덮어쓴다.

In [ ]:
prepare(
    source=path1,
    output=Path("processed/data1")
)

## 4. 일별 결과의 구조 및 기간 확인
생성된 일별 테이블의 컬럼, 지역, 연령 범주와 관측 기간을 확인한다. 업종×연령별로 기록이 있는 날짜 수도 점검한다.

In [ ]:
daily1 = pd.read_csv(
    "processed/data1/daily.csv",
    encoding="utf-8-sig",
    parse_dates=["date"],
    dtype={"region": str, "industry": str, "age": str}
)

print("크기:", daily1.shape)
print("컬럼:", daily1.columns.tolist())
print("지역:", daily1["region"].unique())
print("연령:", daily1["age"].unique())
print("기간:", daily1["date"].min(), "~", daily1["date"].max())
print("관측 날짜 수:", daily1["date"].nunique())

observed_days = daily1.groupby(["industry", "age"])["date"].nunique()

print("업종×연령 조합 수:", len(observed_days))
print("최소 관측일:", observed_days.min())
print("184일 미만 조합 수:", observed_days.lt(184).sum())

## 5. 집계 합계 및 중복 검증
춘천시 개인카드 원본과 일별 결과의 결제금액·결제건수 합계가 일치하는지 확인한다. 일별 결과에 날짜×지역×업종×연령 기준 중복이 없는지도 검사한다.

In [ ]:
for source_col, result_col in [
    ("TS_AT", "amount"),
    ("USE_CNT", "transactions")
]:
    source_total = pd.to_numeric(
        personal1[source_col], errors="raise"
    ).sum()

    print(
        result_col, "집계 전후 합계 일치:",
        source_total == daily1[result_col].sum()
    )

print(
    "일별 집계 키 중복:",
    daily1.duplicated(["date", "region", "industry", "age"]).sum()
)

## 6. 업종·연령 목록과 관측일 표
업종과 연령 범주를 확인하고, 업종×연령별 관측 날짜 수를 표로 정리한다.

표의 0은 해당 조합의 기록이 없다는 뜻이며, 결제금액이나 결제건수가 0이라는 뜻은 아니다.

In [ ]:
# 1. 업종 목록
industries = sorted(daily1["industry"].unique())

print("업종 수:", len(industries))
print("업종 목록:")
print(industries)

# 2. 연령대 목록
age_order = ["10대", "20대", "30대", "40대", "50대", "60대 이상"]
ages = [age for age in age_order if age in daily1["age"].unique()]

print("\n연령대 수:", len(ages))
print("연령대 목록:", ages)

# 3. 업종×연령별 일별 행 수
combination_counts = (
    pd.crosstab(daily1["industry"], daily1["age"])
    .reindex(columns=ages, fill_value=0)
)

print("\n업종×연령별 일별 행 수:")
display(combination_counts)

## 7. 업종·연령별 관측 기간 점검
각 업종×연령의 관측일 수, 최초·최종 관측일과 전체 184일 대비 관측 비율을 계산한다.

행이 없는 날짜는 거래 없음과 비공개·누락을 구분할 수 없어 0으로 채우지 않는다. 관측 비율은 세부 시간대·성별 기록의 완전성을 보장하지 않는다.

In [ ]:
coverage = (
    daily1.groupby(["industry", "age"])
    .agg(
        observed_days=("date", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max")
    )
    .reset_index()
)

coverage["missing_days"] = 184 - coverage["observed_days"]
coverage["coverage_rate"] = coverage["observed_days"] / 184

# 실제 매출 수치 없이 관측일 분포만 확인
print(coverage["observed_days"].describe())

## 8. 업종·연령별 일별 기초통계
결제금액의 합계·평균·중앙값·표준편차·최솟값·최댓값과 결제건수의 합계·평균을 계산한다. 화면에는 앞 12행만 표시하며 전체 결과는 `basic_stats`에 유지한다.

평균은 기록이 있는 날짜 기준이다. 관측일이 1일인 그룹의 표본 표준편차는 계산되지 않아 `NaN`으로 표시된다. 금액 단위는 제공 자료의 정의를 따른다.

In [ ]:
basic_stats = (
    daily1.groupby(["industry", "age"])
    .agg(
        observed_days=("date", "nunique"),
        amount_total=("amount", "sum"),
        amount_mean=("amount", "mean"),
        amount_median=("amount", "median"),
        amount_std=("amount", "std"),
        amount_min=("amount", "min"),
        amount_max=("amount", "max"),
        transactions_total=("transactions", "sum"),
        transactions_mean=("transactions", "mean")
    )
    .reset_index()
)

with pd.option_context(
    "display.float_format", "{:,.1f}".format,
    "display.max_columns", None
):
    display(basic_stats.head(12))

## 2025년 7~12월 공휴일 목록 정리 완료

- 출처: 한국천문연구원 특일정보 API의 공휴일 정보 조회
- 조회 기간: 2025년 7~12월
- 조회일: YYYY-MM-DD
- 선정 기준: isHoliday = Y
- 공휴일명은 API 응답 그대로 기록
- 카드 데이터와 결합하지 않고 별도 목록으로 정리

| 날짜 | 공휴일명 |
|---|---|
| 2025-08-15 | 광복절 |
| 2025-10-03 | 개천절 |
| 2025-10-05 | 추석 |
| 2025-10-06 | 추석 |
| 2025-10-07 | 추석 |
| 2025-10-08 | 대체공휴일 |
| 2025-10-09 | 한글날 |
| 2025-12-25 | 기독탄신일 |

총 8일이며, 7월·9월·11월에는 조회된 공휴일이 없습니다.
일반 토·일요일을 모두 포함한 목록은 아닙니다.
10월 5일은 일요일과 겹치지만 API에서 공휴일로 반환되어 포함했습니다.

출처 안내: https://www.data.go.kr/data/15012690/openapi.do

## 점검 결과 및 기초통계 저장

In [ ]:
output_dir = Path("processed/data1")
output_dir.mkdir(parents=True, exist_ok=True)

basic_stats.to_csv(
    output_dir / "basic_stats.csv",
    index=False,
    encoding="utf-8-sig"
)

coverage.to_csv(
    output_dir / "coverage.csv",
    index=False,
    encoding="utf-8-sig"
)

print("기초통계와 관측일 점검표 저장 완료")